In [2]:
import pandas as pd
import numpy as np



### Finding missing values
- Empty box = NaN. A CSV's empty cell becomes NaN; a column with NaN becomes decimals.
- `df.isna()` -> True/False table (True = missing). `isnull()` is the same. `notna()` is the opposite.
- `df.isna().sum()` -> missing count per column (run this first on any new data).
- `df.isna().sum().sum()` -> total missing in the table.
- `(df.isna().mean() * 100).round(1)` -> percent missing per column.
- `df[df["col"].isna()]` -> rows where col is missing; `df[df["col"].notna()]` -> rows where it is filled.
- `df[df.isna().any(axis=1)]` -> rows with at least one missing value.
- `df.info()` -> Non-Null Count lower than the row count means missing values.
- `value_counts()` skips NaN; `value_counts(dropna=False)` shows it.
- Watch out: `df["x"] == np.nan` is always False (no error); use `isna()`. `df.isna.sum()` without () gives AttributeError.

In [19]:
df = pd.DataFrame({"name":   ["Asha", "Ravi", "Meena", "Kiran", "Dev", "Zoya"],
                   "city":   ["Indore", None, "Pune", "Delhi", None, "Pune"],
                   "age":    [25, 31, None, 28, None, 40],
                   "salary": [50000, 62000, 45000, None, 58000, 71000]})

print(df)

print(df.isna())

print(df.isna().sum())
print(df.isna().sum().sum())
print(df.notna().sum())
print(df.notna().sum().sum())

print(df["age"].isna())
print(df["age"].isna().sum())
print(df[df["age"].isna()])
print(df[df["city"].notna()])
print(df[df.isna().any(axis=1)])

# step 4: percent missing per column
print((df.isna().mean() * 100).round(1))

df.info()

print(df["city"].value_counts())                 # NaN is skipped
print(df["city"].value_counts(dropna=False))


    name    city   age   salary
0   Asha  Indore  25.0  50000.0
1   Ravi    None  31.0  62000.0
2  Meena    Pune   NaN  45000.0
3  Kiran   Delhi  28.0      NaN
4    Dev    None   NaN  58000.0
5   Zoya    Pune  40.0  71000.0
    name   city    age  salary
0  False  False  False   False
1  False   True  False   False
2  False  False   True   False
3  False  False  False    True
4  False   True   True   False
5  False  False  False   False
name      0
city      2
age       2
salary    1
dtype: int64
5
name      6
city      4
age       4
salary    5
dtype: int64
19
0    False
1    False
2     True
3    False
4     True
5    False
Name: age, dtype: bool
2
    name  city  age   salary
2  Meena  Pune  NaN  45000.0
4    Dev  None  NaN  58000.0
    name    city   age   salary
0   Asha  Indore  25.0  50000.0
2  Meena    Pune   NaN  45000.0
3  Kiran   Delhi  28.0      NaN
5   Zoya    Pune  40.0  71000.0
    name   city   age   salary
1   Ravi   None  31.0  62000.0
2  Meena   Pune   NaN  45000.0
3

### Removing and filling missing values
- Both `dropna` and `fillna` return a new table; save the result (`df = df.dropna()`), or the original stays unchanged.
- `dropna()` drops rows with ANY blank. `dropna(subset=["a","b"])` looks only at those columns.
- `how="all"` = drop only if every box is blank. `thresh=n` = keep rows with at least n filled boxes. `axis=1` = drop columns.
- After dropping, fix labels with `.reset_index(drop=True)`.
- `fillna(0)`, `fillna("Unknown")`, `fillna(df["x"].mean())`, `fillna(df["x"].median())`.
- Text: `fillna(df["c"].mode()[0])` (most common value).
- Several columns at once: `df.fillna({"city": "Unknown", "age": 0})`.
- By group: `df["x"].fillna(df.groupby("g")["x"].transform("mean"))`.
- Mean/median skip the blanks themselves. Median is safer with extreme values.
- After filling a number column, `.astype(int)` brings back whole numbers.
- Watch out: not saving the result; `df.fillna(df.mean())` fails when text columns exist (choose the column).

In [ ]:
df.dropna() # drop every row that has ANY blank


,name,city,age,salary
0,Asha,Indore,25.0,50000.0
5,Zoya,Pune,40.0,71000.0


In [21]:
df.dropna(subset=["city"])

,name,city,age,salary
0,Asha,Indore,25.0,50000.0
2,Meena,Pune,NaN,45000.0
3,Kiran,Delhi,28.0,NaN
5,Zoya,Pune,40.0,71000.0


In [22]:
df.dropna(subset=["age", "salary"])

,name,city,age,salary
0,Asha,Indore,25.0,50000.0
1,Ravi,None,31.0,62000.0
5,Zoya,Pune,40.0,71000.0


In [23]:
df.dropna(how="all")      # drop a row only if ALL its boxes are blank (here: nothing is dropped)
df.dropna(thresh=3)       # keep rows that have at least 3 filled boxes (drops Dev, who has only 2)
df.dropna(axis=1)         # drop COLUMNS that have a blank (only 'name' is left)

,name
0,Asha
1,Ravi
2,Meena
3,Kiran
4,Dev
5,Zoya


In [24]:
df.dropna().reset_index(drop=True)

,name,city,age,salary
0,Asha,Indore,25.0,50000.0
1,Zoya,Pune,40.0,71000.0


In [27]:
d = df.copy()
d["age"] = d["age"].fillna(0)
print(d)
d["city"] = d["city"].fillna("Unknown")
print(d)

    name    city   age   salary
0   Asha  Indore  25.0  50000.0
1   Ravi    None  31.0  62000.0
2  Meena    Pune   0.0  45000.0
3  Kiran   Delhi  28.0      NaN
4    Dev    None   0.0  58000.0
5   Zoya    Pune  40.0  71000.0
    name     city   age   salary
0   Asha   Indore  25.0  50000.0
1   Ravi  Unknown  31.0  62000.0
2  Meena     Pune   0.0  45000.0
3  Kiran    Delhi  28.0      NaN
4    Dev  Unknown   0.0  58000.0
5   Zoya     Pune  40.0  71000.0


In [28]:

d = df.copy()
d["age"] = d["age"].fillna(d["age"].mean())          # mean of age = 31.0
d["salary"] = d["salary"].fillna(d["salary"].median())  # median of salary = 58000.0
print(d)

    name    city   age   salary
0   Asha  Indore  25.0  50000.0
1   Ravi    None  31.0  62000.0
2  Meena    Pune  31.0  45000.0
3  Kiran   Delhi  28.0  58000.0
4    Dev    None  31.0  58000.0
5   Zoya    Pune  40.0  71000.0


In [29]:
df["city"].mode()[0]       # 'Pune' -> the most common city. [0] picks the first one.
d["city"] = d["city"].fillna(df["city"].mode()[0])

In [30]:
d = df.fillna({"city": "Unknown", "age": df["age"].median(), "salary": 0})
print(d)

    name     city   age   salary
0   Asha   Indore  25.0  50000.0
1   Ravi  Unknown  31.0  62000.0
2  Meena     Pune  29.5  45000.0
3  Kiran    Delhi  28.0      0.0
4    Dev  Unknown  29.5  58000.0
5   Zoya     Pune  40.0  71000.0


### Fixing types
- A numeric-looking column with Dtype `str` (or `object`) is text: sum glues text (no error), mean/compare go wrong. Check `df.dtypes` / `df.info()` first.
- `astype(int)`, `astype(float)`, `astype(str)`, `astype("category")`; save the result back to the column.
- Float to int cuts off decimals (3.7 -> 3), it does not round.
- One bad entry makes `astype(int)` raise ValueError.
- `pd.to_numeric(df["x"], errors="coerce")` turns bad entries into NaN; then `isna().sum()` shows how many, then fillna/dropna.
- Remove symbols with `.str.replace(...)` before converting ("Rs 1,200" -> 1200).
- Whole-number columns cannot hold NaN: `astype(int)` gives IntCastingNaNError. Fill first, or use `astype("Int64")`.
- Recipe: to_numeric(coerce) -> count NaN -> fill/drop -> astype(int).

In [31]:
raw = pd.DataFrame({"item":  ["pen", "book", "bag", "ball"],
                    "price": ["100", "250", "abc", "80"],     # numbers stored as text, one bad entry
                    "qty":   [2, 5, 1, 3],                    # real numbers
                    "size":  ["10", "20", "30", "40"]})       # numbers stored as text
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   item    4 non-null      object
 1   price   4 non-null      object
 2   qty     4 non-null      int64 
 3   size    4 non-null      object
dtypes: int64(1), object(3)
memory usage: 260.0+ bytes


In [33]:
print(raw["qty"].sum() )     # 11            -> real numbers are added
raw["size"].sum()     # '10203040'    -> text is glued together, not added!

11


'10203040'

In [34]:
r = raw.copy()
r["size"] = r["size"].astype(int)
print(r["size"].dtype)                  # int64
print(r["size"].sum())   


int32
100


In [36]:
a = pd.to_numeric(raw["price"], errors="coerce")
print(a.sum())

430.0


### Duplicates
- `df.duplicated()` -> True for a row that repeats an earlier row (the first one is False). It compares ALL columns unless you give `subset`.
- A unique id column (like `order_id`) makes every row different, so use `subset=[...]` for the columns that define a repeat.
- `df.duplicated().sum()` -> how many repeats. `df[df.duplicated(subset=[...])]` -> the repeated rows.
- `keep=False` in `duplicated` shows the original and all its repeats together.
- `df.drop_duplicates(subset=[...], keep="first")`: `"first"` (default), `"last"`, or `False` (drop every row that has a repeat).
- `ignore_index=True` (or `reset_index(drop=True)`) renumbers the labels.
- `df["c"].nunique()` = number of different values, `df["c"].unique()` = the values.
- Clean text first (`.str.strip().str.lower()`), otherwise "Asha" and "asha " look different.
- Watch out: not saving the result; wrong column in `subset` -> KeyError; no `subset` with an id column finds nothing.

In [38]:
 d = pd.DataFrame({"order_id": [1, 2, 3, 4, 5, 6],
                  "customer": ["asha", "ravi", "asha", "meena", "ravi", "asha"],
                  "item":     ["pen", "book", "pen", "bag", "book", "ball"],
                  "amount":   [100, 250, 100, 400, 250, 90]})

In [40]:
d.duplicated()

0    False
1    False
2    False
3    False
4    False
5    False
dtype: bool

In [41]:
d.duplicated(subset=["customer", "item"])

0    False
1    False
2     True
3    False
4     True
5    False
dtype: bool

In [43]:
d[d.duplicated(subset=["customer","item"])]

,order_id,customer,item,amount
2,3,asha,pen,100
4,5,ravi,book,250


In [44]:
d[d.duplicated(subset=["customer","item"],keep=False)]

,order_id,customer,item,amount
0,1,asha,pen,100
1,2,ravi,book,250
2,3,asha,pen,100
4,5,ravi,book,250


In [45]:
d.drop_duplicates(subset=["customer", "item"])

,order_id,customer,item,amount
0,1,asha,pen,100
1,2,ravi,book,250
3,4,meena,bag,400
5,6,asha,ball,90


In [46]:
d.drop_duplicates(subset=["customer", "item"]).reset_index()

,index,order_id,customer,item,amount
0,0,1,asha,pen,100
1,1,2,ravi,book,250
2,3,4,meena,bag,400
3,5,6,asha,ball,90


In [53]:
e = pd.DataFrame({"a": [1, 2, 1, 2], "b": ["x", "y", "x", "y"]})
print(e.shape)
print(e["a"].sum())
a = e.drop_duplicates
len(e), len(e.drop_duplicates())    

(4, 2)
6


(4, 2)

In [54]:
d["customer"].nunique() 

3

In [55]:
d["customer"].unique()    

array(['asha', 'ravi', 'meena'], dtype=object)

In [56]:
d["customer"].drop_duplicates() 

0     asha
1     ravi
3    meena
Name: customer, dtype: object